# Environment Setup

Import required libraries and initialize project configuration for monitoring.

In [1]:
# Import libraries and configure project settings

import warnings
warnings.filterwarnings("ignore")

import json
import boto3
import sagemaker
import pandas as pd
import numpy as np

PROJECT_PREFIX = "loanwise"

DASHBOARD_NAME = "LoanApprovalMonitoringDashboard"

sess = sagemaker.Session()

bucket = sess.default_bucket()

cloudwatch = boto3.client("cloudwatch")

print(f"SageMaker Version : {sagemaker.__version__}")
print(f"Bucket            : {bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


SageMaker Version : 2.224.4
Bucket            : sagemaker-us-east-1-612256167011


# Load Monitoring Datasets

Load baseline training data and inference outputs.

In [2]:
# Load monitoring datasets

train_df = pd.read_csv(
    "processed_train.csv"
)

predictions_df = pd.read_csv(
    "predictions.csv"
)

print(f"Training Shape    : {train_df.shape}")
print(f"Predictions Shape : {predictions_df.shape}")

display(predictions_df.head())

Training Shape    : (16000, 18)
Predictions Shape : (500, 3)


,Actual,Prediction,Probability
0,0,1,0.637472
1,1,1,0.808633
2,1,1,0.941834
3,1,1,0.639328
4,0,0,0.207672


# Data Quality Monitoring

Analyze missing values and duplicate records.

In [3]:
# Generate data quality metrics

data_quality_report = {
    "total_records": int(len(train_df)),
    "missing_values": int(
        train_df.isnull().sum().sum()
    ),
    "duplicate_records": int(
        train_df.duplicated().sum()
    )
}

display(data_quality_report)

{'total_records': 16000, 'missing_values': 0, 'duplicate_records': 0}

# Data Drift Monitoring

Compare baseline and inference feature distributions.

In [4]:
# Calculate simple data drift metrics

baseline_income = train_df[
    "AMT_INCOME_TOTAL"
].mean()

current_income = train_df[
    "AMT_INCOME_TOTAL"
].sample(
    n=min(500, len(train_df)),
    random_state=42
).mean()

income_drift_percent = round(
    abs(
        current_income - baseline_income
    )
    / baseline_income
    * 100,
    2
)

data_drift_report = {
    "baseline_income_mean": float(
        baseline_income
    ),
    "current_income_mean": float(
        current_income
    ),
    "drift_percentage": float(
        income_drift_percent
    )
}

display(data_drift_report)

{'baseline_income_mean': 165017.19121875,
 'current_income_mean': 163759.752,
 'drift_percentage': 0.76}

# Model Performance Monitoring

Review model performance using prediction outputs.

In [5]:
# Load prediction summary

with open(
    "prediction_summary.json",
    "r"
) as f:

    model_performance_report = json.load(f)

display(model_performance_report)

{'records_scored': 500,
 'accuracy': 0.678,
 'precision': 0.6872427983539094,
 'recall': 0.6626984126984127,
 'f1_score': 0.6747474747474748,
 'roc_auc': 0.6781233998975934}

# Generate Monitoring Reports

Save monitoring reports locally.

In [6]:
# Save monitoring reports

with open(
    "data_quality_report.json",
    "w"
) as f:

    json.dump(
        data_quality_report,
        f,
        indent=4
    )

with open(
    "data_drift_report.json",
    "w"
) as f:

    json.dump(
        data_drift_report,
        f,
        indent=4
    )

with open(
    "model_performance_report.json",
    "w"
) as f:

    json.dump(
        model_performance_report,
        f,
        indent=4
    )

print("Monitoring reports created.")

Monitoring reports created.


# Create CloudWatch Dashboard

Create an infrastructure monitoring dashboard in CloudWatch.

In [7]:
# Create CloudWatch dashboard

dashboard_body = {
    "widgets": [
        {
            "type": "text",
            "x": 0,
            "y": 0,
            "width": 24,
            "height": 6,
            "properties": {
                "markdown": (
                    "# Loan Approval Monitoring Dashboard\n"
                    "Data Monitoring\n"
                    "Model Monitoring\n"
                    "Infrastructure Monitoring"
                )
            }
        }
    ]
}

cloudwatch.put_dashboard(
    DashboardName=DASHBOARD_NAME,
    DashboardBody=json.dumps(
        dashboard_body
    )
)

print(
    f"Dashboard '{DASHBOARD_NAME}' created."
)

Dashboard 'LoanApprovalMonitoringDashboard' created.


# Upload Monitoring Artifacts

Upload monitoring reports to Amazon S3.

In [8]:
# Upload monitoring reports

quality_s3_uri = sess.upload_data(
    path="data_quality_report.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/monitoring"
)

drift_s3_uri = sess.upload_data(
    path="data_drift_report.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/monitoring"
)

performance_s3_uri = sess.upload_data(
    path="model_performance_report.json",
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/monitoring"
)

print(quality_s3_uri)
print(drift_s3_uri)
print(performance_s3_uri)

s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/data_quality_report.json
s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/data_drift_report.json
s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/model_performance_report.json


# Verify Monitoring Outputs

Review generated reports and CloudWatch assets.

In [9]:
# Verify monitoring outputs

print("Generated Reports")
print("-----------------")
print("data_quality_report.json")
print("data_drift_report.json")
print("model_performance_report.json")

print("\nCloudWatch Dashboard")
print("--------------------")
print(DASHBOARD_NAME)

print("\nAmazon S3 Artifacts")
print("-------------------")
print(quality_s3_uri)
print(drift_s3_uri)
print(performance_s3_uri)

Generated Reports
-----------------
data_quality_report.json
data_drift_report.json
model_performance_report.json

CloudWatch Dashboard
--------------------
LoanApprovalMonitoringDashboard

Amazon S3 Artifacts
-------------------
s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/data_quality_report.json
s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/data_drift_report.json
s3://sagemaker-us-east-1-612256167011/loanwise/monitoring/model_performance_report.json


# Summary and Next Steps

Notebook 5 successfully generated monitoring reports and infrastructure monitoring assets.

Outputs generated:

- data_quality_report.json
- data_drift_report.json
- model_performance_report.json
- LoanApprovalMonitoringDashboard

These outputs will be used in Notebook 6 to demonstrate CI/CD automation and operational validation.

In [10]:
# Notebook completion

print("Notebook 5 completed successfully.")

Notebook 5 completed successfully.
